# 🧪 PT-W1-D2 概念实验：Context Map 依赖方向

> 配套阅读：同名 .md
> 实验目标：用有向图模拟 Context Map，检验 Context 间依赖关系

## 第 1 格：用邻接表模拟 Context Map

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Tuple

REL_TYPES = {
    "C-S": "Customer-Supplier（下游依赖上游，有契约）",
    "SK":  "Shared Kernel（共享子模型）",
    "CF":  "Conformist（下游无条件遵从）",
    "ACL": "Anti-Corruption Layer（防腐层隔离）",
}

context_map = {
    "Lease Management": [
        ("Contract Management", "SK", "shares_lease_concept"),
        ("Space Management", "C-S", "occupies"),
    ],
    "Contract Management": [
        ("Tenant Management", "C-S", "is_signed_by"),
        ("Space Management", "C-S", "occupies"),
    ],
    "Billing Management": [
        ("Contract Management", "C-S", "reads_clauses"),
        ("Payment Management", "C-S", "drives_collection"),
    ],
    "Payment Management": [
        ("Finance Settlement", "C-S", "settles_to"),
    ],
    "Customer Management": [
        ("Tenant Management", "CF", "converts_to"),
    ],
    "Property Service": [
        ("Space Management", "C-S", "maintains"),
    ],
}

print("Context Map — Context 间关系：")
for src, rels in context_map.items():
    for tgt, rel_type, verb in rels:
        desc = REL_TYPES.get(rel_type, rel_type)
        print(f"  {src} --[{verb}]--({desc})--> {tgt}")

## 第 2 格：检验边界问题 — Contract vs Lease

In [ ]:
def check_boundary_leak(context_name, allowed_objects, actual_imports):
    leaks = [obj for obj in actual_imports if obj not in allowed_objects]
    status = "✅ 边界安全" if not leaks else f"❌ 泄漏: {leaks}"
    print(f"[{context_name}] imports={actual_imports} → {status}")

check_boundary_leak("Contract Management",
    ["Contract", "ContractClause", "ContractTemplate"],
    ["Contract", "ContractClause", "RentMethod"])

check_boundary_leak("Lease Management",
    ["Occupancy", "OccupancyPeriod", "LeaseStatus"],
    ["Occupancy", "ContractApprovalStatus"])

print("\n⚠️ 发现泄漏说明需要 Context Map 显式声明翻译层")

## 第 3 格：可视化 Context Map 有向图

In [ ]:
from matplotlib import font_manager, pyplot as plt
import numpy as np

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("字体:", font_name)
import matplotlib.patches as mpatches

ctx_pos = {
    "Lease": (1, 3), "Contract": (3, 3), "Tenant": (5, 3),
    "Space": (1, 1), "Billing": (3, 1), "Payment": (5, 1),
    "Finance": (7, 1), "Customer": (7, 3), "PropertySvc": (1, 5),
}

fig, ax = plt.subplots(figsize=(10, 8))
for name, (x, y) in ctx_pos.items():
    is_core = name in ("Lease", "Contract", "Tenant", "Space", "Billing", "Payment")
    color = "#e74c3c" if is_core else "#3498db"
    circle = plt.Circle((x, y), 0.4, color=color, alpha=0.8)
    ax.add_patch(circle)
    ax.text(x, y, name, ha="center", va="center", fontsize=9, fontweight="bold")

edges = [
    ("Lease", "Contract", "SK"), ("Lease", "Space", "occupies"),
    ("Contract", "Tenant", "signs"), ("Contract", "Space", "occupies"),
    ("Billing", "Contract", "reads"), ("Billing", "Payment", "drives"),
    ("Payment", "Finance", "settles"), ("Customer", "Tenant", "converts"),
    ("PropertySvc", "Space", "maintains"),
]
for src, tgt, label in edges:
    sx, sy = ctx_pos[src]; tx, ty = ctx_pos[tgt]
    ax.annotate("", xy=(tx, ty), xytext=(sx, sy),
                arrowprops=dict(arrowstyle="->", color="#2c3e50", lw=1.5))
    mx, my = (sx+tx)/2, (sy+ty)/2
    ax.text(mx, my + 0.15, label, fontsize=7, ha="center", color="#7f8c8d")

ax.set_xlim(-0.5, 8.5); ax.set_ylim(-0.5, 6)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("MI Context Map — Context 间依赖关系", fontsize=14)
plt.tight_layout()
plt.savefig("/tmp/w1d2_context_map.png", dpi=120)
plt.show()
print("结论：Context 划分 80% 正确，但关系是隐式的 → 需要 Context Map 显式声明")